# Ficheros · Intermedio

Procesar ficheros CSV/TSV y persistir datos en ficheros de texto y en SQLite.

Los ficheros `cotizacion.csv` y `calificaciones.csv` son los originales de los ejercicios de [aprendeconalf.es](https://aprendeconalf.es/python-ejercicios/) y se descargan desde su repositorio.

## 1. PIB per cápita de la UE

Leer el fichero de Eurostat con el PIB per cápita de los países europeos (conjunto `sdg_08_10`), preguntar las iniciales de un país y mostrar su PIB per cápita en todos los años disponibles.

In [1]:
from urllib import request

URL_PIB = "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/sdg_08_10?format=TSV&compressed=false"

def pib(url, pais):
    with request.urlopen(url) as f:
        cabecera, *filas = f.read().decode("utf-8").splitlines()
    años = [a.strip() for a in cabecera.split("\t")[1:]]
    for fila in filas:
        claves, *valores = fila.split("\t")
        # CLV20_EUR_HAB = euros por habitante a precios constantes
        if claves.endswith("," + pais) and "EUR_HAB" in claves:
            print(claves)
            return {a: v.strip() for a, v in zip(años, valores)}
    return {}

for año, valor in pib(URL_PIB, input("Iniciales del país: ").upper()).items():
    print(año, valor)

Iniciales del país: ES
A,CLV20_EUR_HAB,B1GQ,ES
2000 22930
2001 23700
2002 23970
2003 24220
2004 24590
2005 25000
2006 25600
2007 25990
2008 25760
2009 24590
2010 24510
2011 24260
2012 23560
2013 23310
2014 23730
2015 24720
2016 25420
2017 26110
2018 26620
2019 26930
2020 23850
2021 25440
2022 26820
2023 27130
2024 27820 p
2025 28260 p


## 2. Listín telefónico

Gestionar un listín en `listin.txt` (una línea `nombre,teléfono` por cliente) con funciones para crearlo si no existe, consultar el teléfono de un cliente, añadir un cliente y eliminarlo.

In [2]:
import os

LISTIN = "listin.txt"

def crea_listin():
    if not os.path.exists(LISTIN):
        open(LISTIN, "w", encoding="utf-8").close()

def lee_listin():
    with open(LISTIN, encoding="utf-8") as f:
        return dict(linea.strip().split(",") for linea in f if linea.strip())

def guarda_listin(listin):
    with open(LISTIN, "w", encoding="utf-8") as f:
        f.writelines(f"{nombre},{telefono}\n" for nombre, telefono in listin.items())

def consulta(nombre):
    return lee_listin().get(nombre, "No está en el listín")

def añade(nombre, telefono):
    listin = lee_listin()
    listin[nombre] = telefono
    guarda_listin(listin)

def elimina(nombre):
    listin = lee_listin()
    listin.pop(nombre, None)
    guarda_listin(listin)

crea_listin()
añade("Ana", "600111222")
añade("Luis", "700333444")
print(consulta("Ana"))
elimina("Ana")
print(consulta("Ana"), lee_listin())

600111222
No está en el listín {'Luis': '700333444'}


## 3. Cotizaciones del IBEX35

`cotizacion.csv` tiene las columnas Nombre, Final, Máximo, Mínimo, Volumen y Efectivo, separadas por `;` y con números en formato español. Escribir una función que lea el fichero y devuelva un diccionario por columnas, y otra que cree un CSV con el mínimo, el máximo y la media de cada columna numérica.

In [3]:
from urllib import request

URL_DATOS = "https://raw.githubusercontent.com/asalber/aprendeconalf/master/content/es/docencia/python/ejercicios/soluciones/ficheros/"
request.urlretrieve(URL_DATOS + "cotizacion.csv", "cotizacion.csv")

def a_numero(cifra):
    # 1.234,5 -> 1234.5
    return float(cifra.replace(".", "").replace(",", "."))

def lee_cotizaciones(ruta):
    with open(ruta, encoding="utf-8") as f:
        claves, *filas = [linea.strip().split(";") for linea in f if linea.strip()]
    columnas = {clave: [] for clave in claves}
    for fila in filas:
        columnas[claves[0]].append(fila[0])
        for clave, valor in zip(claves[1:], fila[1:]):
            columnas[clave].append(a_numero(valor))
    return columnas

def resumen(columnas, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        f.write("Columna;Mínimo;Máximo;Media\n")
        for clave, valores in list(columnas.items())[1:]:
            f.write(f"{clave};{min(valores)};{max(valores)};{sum(valores) / len(valores):.2f}\n")

cotizaciones = lee_cotizaciones("cotizacion.csv")
print(cotizaciones["Nombre"][:5])
resumen(cotizaciones, "resumen-cotizacion.csv")
print(open("resumen-cotizacion.csv", encoding="utf-8").read())

['ACCIONA', 'ACERINOX', 'ACS', 'AENA', 'AMADEUS']
Columna;Mínimo;Máximo;Media
Final;1.0165;19705.0;2796.77
Máximo;4.0675;19875.0;3170.11
Mínimo;1.0165;19675.0;3136.51
Volumen;1221.0;36129692.0;4252278.51
Efectivo;2343.09;145765.44;31767.78



## 4. Calificaciones de un curso

`calificaciones.csv` contiene apellidos, nombre, asistencia, dos parciales de teoría, sus repeticiones en la convocatoria ordinaria, prácticas y su repetición. Escribir funciones que:

1. Lean el fichero y devuelvan una lista de diccionarios (uno por alumno) ordenada por apellidos.
2. Añadan a cada alumno la nota final: 30 % cada parcial y 40 % prácticas (si repitió un examen, cuenta la nota de la ordinaria).
3. Devuelvan dos listas: aprobados y suspensos. Para aprobar: asistencia ≥ 75 %, parciales y prácticas ≥ 4 y nota final ≥ 5.

In [4]:
request.urlretrieve(URL_DATOS + "calificaciones.csv", "calificaciones.csv")

def lee_calificaciones(ruta):
    with open(ruta, encoding="utf-8") as f:
        claves, *filas = [linea.strip().split(";") for linea in f if linea.strip()]
    alumnos = []
    for fila in filas:
        alumno = dict(zip(claves, fila))
        alumno["Asistencia"] = int(alumno["Asistencia"].rstrip("%"))
        for clave in claves[3:]:
            alumno[clave] = float(alumno[clave].replace(",", ".")) if alumno[clave] else 0.0
        alumnos.append(alumno)
    return sorted(alumnos, key=lambda a: a["Apellidos"])

def nota(alumno, examen, ordinario):
    # la nota de la ordinaria sustituye a la del examen
    return alumno[ordinario] or alumno[examen]

def añade_nota_final(alumnos):
    for a in alumnos:
        a["Parcial1"] = nota(a, "Parcial1", "Ordinario1")
        a["Parcial2"] = nota(a, "Parcial2", "Ordinario2")
        a["Practicas"] = nota(a, "Practicas", "OrdinarioPracticas")
        a["Final"] = round(0.3 * a["Parcial1"] + 0.3 * a["Parcial2"] + 0.4 * a["Practicas"], 2)
    return alumnos

def aprobados_suspensos(alumnos):
    aprobados, suspensos = [], []
    for a in alumnos:
        aprueba = a["Asistencia"] >= 75 and min(a["Parcial1"], a["Parcial2"], a["Practicas"]) >= 4 and a["Final"] >= 5
        (aprobados if aprueba else suspensos).append(f"{a['Apellidos']}, {a['Nombre']} ({a['Final']})")
    return aprobados, suspensos

aprobados, suspensos = aprobados_suspensos(añade_nota_final(lee_calificaciones("calificaciones.csv")))
print("Aprobados:", aprobados)
print("Suspensos:", suspensos)

Aprobados: ['Bueno Cerdeira, Patricia (8.05)', 'Casariego García, Raúl (5.0)', 'Curbelo Sánchez, Jorge (5.9)', 'Díaz Souto, Sofía (7.65)', 'García Perez, Yaiza (8.4)', 'Lorenzo García, Jaime (8.4)', 'Martínez Lucas, Cecilia (5.8)', 'Moreno Angulo, Antonio (6.82)', 'Morillo Escudero, Ana (9.35)', 'Muñoz Gómez, Carolina (5.88)', 'Rodríguez de Blas, Ignacio (6.65)', 'Sánchez Jordán, María (7.92)']
Suspensos: ['Anido Bonet, David (3.15)', 'Mora Peñaloza, Sandra (5.38)', 'Ramirez de la Puente, Raquel (2.42)', 'Riego Pizarro, Carlos (2.62)']


## 5. Gastos en SQLite

Guardar gastos en una base de datos SQLite con una tabla `gastos` que incluya la fecha del gasto. Implementar las operaciones CRUD (`add_gasto`, `get_gastos`, `update_gasto`, `remove_gasto`) y permitir filtrar por mes.

In [5]:
import sqlite3

con = sqlite3.connect("gastos.db")
con.execute("CREATE TABLE IF NOT EXISTS gastos (id INTEGER PRIMARY KEY, concepto TEXT, cantidad REAL, fecha TEXT)")

def add_gasto(concepto, cantidad, fecha):
    with con:
        return con.execute("INSERT INTO gastos (concepto, cantidad, fecha) VALUES (?, ?, ?)", (concepto, cantidad, fecha)).lastrowid

def get_gastos(mes=None):
    # mes con formato 'aaaa-mm'
    if mes:
        return con.execute("SELECT * FROM gastos WHERE strftime('%Y-%m', fecha) = ?", (mes,)).fetchall()
    return con.execute("SELECT * FROM gastos").fetchall()

def update_gasto(id_, cantidad):
    with con:
        con.execute("UPDATE gastos SET cantidad = ? WHERE id = ?", (cantidad, id_))

def remove_gasto(id_):
    with con:
        con.execute("DELETE FROM gastos WHERE id = ?", (id_,))

alquiler = add_gasto("Alquiler", 800, "2026-10-01")
luz = add_gasto("Luz", 65.4, "2026-10-15")
add_gasto("Comida", 120, "2026-11-02")
print(get_gastos())
print(get_gastos("2026-10"))
update_gasto(luz, 70)
remove_gasto(alquiler)
print(get_gastos())
con.close()

[(1, 'Alquiler', 800.0, '2026-10-01'), (2, 'Luz', 65.4, '2026-10-15'), (3, 'Comida', 120.0, '2026-11-02')]
[(1, 'Alquiler', 800.0, '2026-10-01'), (2, 'Luz', 65.4, '2026-10-15')]
[(2, 'Luz', 70.0, '2026-10-15'), (3, 'Comida', 120.0, '2026-11-02')]
